🔹 Cell 1: Imports

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

🔹 Cell 2: Load Data (same as load_data())

In [2]:
DATA_PATH = "training_data.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Training data not found at {DATA_PATH}")

df = pd.read_csv(DATA_PATH, parse_dates=["date"])

df.head()

,product_id,category,price,competitor_price,discount,units_sold,date,season,day_of_week,stock_available,marketing_spend,customer_rating
0,P039,Sports,1716.21,1805.14,13.9,53,2025-02-08,Winter,Saturday,348,893.21,4.5
1,P021,Electronics,1051.88,1007.69,0.2,37,2024-11-30,Fall,Saturday,468,132.49,4.0
2,P044,Sports,1107.10,1117.92,6.0,41,2025-08-13,Summer,Wednesday,459,470.80,3.6
3,P017,Sports,409.80,422.11,3.3,64,2024-12-03,Winter,Tuesday,406,746.12,4.3
4,P008,Sports,176.89,162.61,2.0,62,2024-01-15,Winter,Monday,497,75.95,4.5


Cell 3: Basic Check (Light EDA)

In [3]:
print("Shape:", df.shape)
df.info()
df.isnull().sum()

Shape: (10000, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   product_id        10000 non-null  object        
 1   category          10000 non-null  object        
 2   price             10000 non-null  float64       
 3   competitor_price  10000 non-null  float64       
 4   discount          10000 non-null  float64       
 5   units_sold        10000 non-null  int64         
 6   date              10000 non-null  datetime64[ns]
 7   season            10000 non-null  object        
 8   day_of_week       10000 non-null  object        
 9   stock_available   10000 non-null  int64         
 10  marketing_spend   10000 non-null  float64       
 11  customer_rating   10000 non-null  float64       
dtypes: datetime64[ns](1), float64(5), int64(2), object(4)
memory usage: 937.6+ KB


product_id          0
category            0
price               0
competitor_price    0
discount            0
units_sold          0
date                0
season              0
day_of_week         0
stock_available     0
marketing_spend     0
customer_rating     0
dtype: int64

🔹 Cell 4: Handle Missing Values 

In [4]:
# Numeric → median
numeric_cols = df.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# Categorical → mode
cat_cols = df.select_dtypes(include=["object"]).columns

for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

# Verify
df.isnull().sum()

product_id          0
category            0
price               0
competitor_price    0
discount            0
units_sold          0
date                0
season              0
day_of_week         0
stock_available     0
marketing_spend     0
customer_rating     0
dtype: int64

Cell 5: Feature Engineering

In [5]:
# Price difference
df["price_diff"] = df["price"] - df["competitor_price"]

# Price ratio
df["price_ratio"] = df["price"] / df["competitor_price"].clip(lower=1)

# Effective price after discount
df["effective_price"] = df["price"] * (1 - df["discount"] / 100)

# Sort for rolling features
df = df.sort_values(["product_id", "date"]).reset_index(drop=True)

# Rolling average demand
df["rolling_demand_7"] = (
    df.groupby("product_id")["units_sold"]
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)

# Date features
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter

# Weekend flag
df["is_weekend"] = df["day_of_week"].isin(["Saturday", "Sunday"]).astype(int)

# Log marketing spend
df["log_marketing_spend"] = np.log1p(df["marketing_spend"])

df.head()

,product_id,category,price,competitor_price,discount,units_sold,date,season,day_of_week,stock_available,marketing_spend,customer_rating,price_diff,price_ratio,effective_price,rolling_demand_7,month,quarter,is_weekend,log_marketing_spend
0,P001,Beauty,185.14,156.61,10.1,67,2023-01-02,Winter,Monday,427,188.10,3.7,28.53,1.182172,166.44086,67.00,1,1,0,5.242276
1,P001,Electronics,629.28,691.56,6.2,62,2023-01-03,Winter,Tuesday,211,626.59,3.0,-62.28,0.909943,590.26464,64.50,1,1,0,6.441887
2,P001,Home,662.26,648.06,9.7,45,2023-02-06,Winter,Monday,358,209.64,4.6,14.20,1.021912,598.02078,58.00,2,1,0,5.350151
3,P001,Beauty,796.64,783.62,5.1,55,2023-02-20,Winter,Monday,83,2389.42,4.5,13.02,1.016615,756.01136,57.25,2,1,0,7.779224
4,P001,Electronics,1392.93,1308.56,7.3,50,2023-02-22,Winter,Wednesday,486,1150.53,3.9,84.37,1.064475,1291.24611,55.80,2,1,0,7.048847


Cell 6: Encode Categoricals

In [7]:
categorical_cols = ["category", "season", "day_of_week"]
encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[f"{col}_encoded"] = le.fit_transform(df[col])
    encoders[col] = le

Cell 7: Feature Selection

In [8]:
feature_cols = [
    "price",
    "competitor_price",
    "discount",
    "stock_available",
    "marketing_spend",
    "customer_rating",
    "price_diff",
    "price_ratio",
    "effective_price",
    "rolling_demand_7",
    "month",
    "quarter",
    "is_weekend",
    "log_marketing_spend",
    "category_encoded",
    "season_encoded",
    "day_of_week_encoded",
]

TARGET = "units_sold"

X = df[feature_cols]
y = df[TARGET]

🔹 Cell 8: Train-Test Split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (8000, 17)
Test: (2000, 17)


🔹 Cell 9: Final Check

In [11]:
X_train.head()

,price,competitor_price,discount,stock_available,marketing_spend,customer_rating,price_diff,price_ratio,effective_price,rolling_demand_7,month,quarter,is_weekend,log_marketing_spend,category_encoded,season_encoded,day_of_week_encoded
9254,1027.56,953.01,6.5,451,332.27,4.4,74.55,1.078226,960.76860,53.285714,3,1,0,5.808953,3,1,5
1561,258.77,274.18,3.3,113,13.71,3.5,-15.41,0.943796,250.23059,52.857143,4,2,0,2.688528,4,1,4
1670,1287.57,1344.34,7.2,15,401.54,1.9,-56.77,0.957771,1194.86496,54.714286,1,1,0,5.997794,1,3,6
6087,1169.36,1079.86,10.1,23,858.06,4.2,89.50,1.082881,1051.25464,41.285714,11,4,1,6.755839,4,0,3
6669,1644.60,1539.78,12.5,81,76.97,4.4,104.82,1.068075,1439.02500,50.571429,8,3,0,4.356324,4,2,4


In [13]:
import joblib

joblib.dump((X_train, X_test, y_train, y_test), "processed_data.pkl")

['processed_data.pkl']